In [1]:
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os, glob, time
from pathlib import Path

In [2]:
BASE_DIR     = r"D:\program vscode\MoneyLens\ai\Dataset_ocr"
ARRAYS_DIR   = os.path.join(BASE_DIR, "preprocessed")
IMG_H, IMG_W = 32, 128
CHANNELS     = 1
CHARACTERS   = list(
    "0123456789abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ .,:-/()%"
)
NUM_CLASSES  = len(CHARACTERS) + 1
 
# Kelas dataset + prioritas untuk MoneyLens
LABEL_CLASSES = {
    "total_transaksi"  : "🔴 KRITIS  ",   # wajib akurat
    "tanggal"          : "🔴 KRITIS  ",   # wajib akurat
    "total_harga_barang": "🟡 PENTING ",
    "harga_satuan"     : "🟡 PENTING ",
    "QTY"              : "🟢 TAMBAHAN",
    "nama_produk"      : "🟢 TAMBAHAN",
}

In [3]:
def build_model():
    inputs = keras.Input(shape=(IMG_H, IMG_W, CHANNELS))
    x = layers.Conv2D(32, (3,3), padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(64, (3,3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(128, (3,3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,1))(x)
    new_h = IMG_H // 8
    new_w = IMG_W // 4
    x = layers.Reshape((new_w, new_h * 128))(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(64,  return_sequences=True))(x)
    output = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, output)
 
print("=" * 65)
print("TASK 3: EVALUASI PERFORMA PREPROCESSING PER KELAS")
print("=" * 65)
 
model = build_model()
print(f"Model params : {model.count_params():,}")
print()

TASK 3: EVALUASI PERFORMA PREPROCESSING PER KELAS
Model params : 497,672



In [4]:
def pixel_contrast(arr: np.ndarray) -> float:
    """Kontras — semakin tinggi teks semakin jelas"""
    return round(float(arr.std()), 4)
 
def sharpness(arr: np.ndarray) -> float:
    """Ketajaman via Laplacian variance"""
    img_u8 = (arr[:,:,0] * 255).astype(np.uint8)
    return round(float(cv2.Laplacian(img_u8, cv2.CV_64F).var()), 2)
 
def black_pixel_ratio(arr: np.ndarray) -> float:
    """Rasio piksel teks — terlalu tinggi = noise, terlalu rendah = teks hilang"""
    return round(float((arr < 0.5).mean()), 3)
 
def model_confidence(arr: np.ndarray) -> float:
    """Rata-rata confidence model (forward pass tanpa training)"""
    batch = arr.reshape(1, IMG_H, IMG_W, CHANNELS)
    pred  = model.predict(batch, verbose=0)
    return round(float(pred.max(axis=-1).mean()) * 100, 2)
 
def inference_ms(arr: np.ndarray) -> float:
    """Kecepatan inferensi dalam milidetik"""
    batch = arr.reshape(1, IMG_H, IMG_W, CHANNELS)
    t0    = time.perf_counter()
    model.predict(batch, verbose=0)
    return round((time.perf_counter() - t0) * 1000, 1)

In [5]:
print(f"{'Kelas':<22} {'Prioritas':<12} {'N':>4} {'Contrast':>9} "
      f"{'Sharp':>8} {'BPR':>6} {'Conf%':>7} {'ms':>7}")
print("-" * 85)
 
all_results  = []
first_timing = True   # skip pengukuran pertama (warm-up)
 
for split in ["train", "valid", "test"]:
    arrays_dir = os.path.join(ARRAYS_DIR, split, "arrays")
    if not os.path.exists(arrays_dir):
        continue
 
    npy_files = glob.glob(os.path.join(arrays_dir, "*.npy"))
    if not npy_files:
        continue
 
    for npy_path in npy_files:
        # Tentukan kelas dari nama file
        fname = Path(npy_path).stem   # contoh: X123_total_transaksi_00
        cls   = None
        for label in LABEL_CLASSES:
            if label in fname:
                cls = label
                break
        if cls is None:
            continue
 
        try:
            arr = np.load(npy_path)
            if arr.shape != (IMG_H, IMG_W, CHANNELS):
                continue
 
            contrast = pixel_contrast(arr)
            sharp    = sharpness(arr)
            bpr      = black_pixel_ratio(arr)
            conf     = model_confidence(arr)
 
            # Ukur waktu (skip warm-up pertama)
            if first_timing:
                inference_ms(arr)
                first_timing = False
            ms = inference_ms(arr)
 
            all_results.append({
                "split"   : split,
                "file"    : fname,
                "kelas"   : cls,
                "prioritas": LABEL_CLASSES[cls].strip(),
                "contrast": contrast,
                "sharp"   : sharp,
                "bpr"     : bpr,
                "conf_pct": conf,
                "ms"      : ms,
            })
 
        except Exception as e:
            print(f"  [ERROR] {Path(npy_path).name}: {e}")

Kelas                  Prioritas       N  Contrast    Sharp    BPR   Conf%      ms
-------------------------------------------------------------------------------------


In [6]:
df = pd.DataFrame(all_results)
 
if df.empty:
    print("\n[WARNING] Tidak ada data .npy ditemukan.")
    print("          Pastikan Task 2 sudah dijalankan terlebih dahulu!")
else:
    for cls, prio in LABEL_CLASSES.items():
        rows = df[df["kelas"] == cls]
        if rows.empty:
            print(f"  {cls:<22} {prio:<12} {'—':>4}")
            continue
        print(f"  {cls:<22} {prio:<12} "
              f"{len(rows):>4} "
              f"{rows['contrast'].mean():>9.4f} "
              f"{rows['sharp'].mean():>8.2f} "
              f"{rows['bpr'].mean():>6.3f} "
              f"{rows['conf_pct'].mean():>6.1f}% "
              f"{rows['ms'].mean():>7.1f}")

  total_transaksi        🔴 KRITIS      401    0.4645 14442.68  0.444    1.5%   128.8
  tanggal                🔴 KRITIS      383    0.4511 20743.12  0.436    1.5%   129.6
  total_harga_barang     🟡 PENTING    1027    0.4677 13203.35  0.452    1.5%   130.0
  harga_satuan           🟡 PENTING     739    0.4693 12881.90  0.461    1.5%   133.6
  QTY                    🟢 TAMBAHAN    950    0.4762  7992.76  0.492    1.4%   129.4
  nama_produk            🟢 TAMBAHAN   1031    0.4309 39784.58  0.384    1.5%   129.0


In [7]:
print(f"\n{'='*65}")
print("ANALISIS KELAS KRITIS (MoneyLens)")
print(f"{'='*65}")
 
for cls in ["total_transaksi", "tanggal"]:
        rows = df[df["kelas"] == cls]
        if rows.empty:
            print(f"\n  {cls}: tidak ada data")
            continue
        avg_sharp = rows["sharp"].mean()
        avg_conf  = rows["conf_pct"].mean()
        status    = "✅ LAYAK" if avg_sharp > 50 else "⚠️  PERLU PERBAIKAN"
        print(f"\n  {cls}:")
        print(f"    Jumlah sampel : {len(rows)}")
        print(f"    Avg Sharpness : {avg_sharp:.2f}")
        print(f"    Avg Confidence: {avg_conf:.1f}%")
        print(f"    Status        : {status}")


ANALISIS KELAS KRITIS (MoneyLens)

  total_transaksi:
    Jumlah sampel : 401
    Avg Sharpness : 14442.68
    Avg Confidence: 1.5%
    Status        : ✅ LAYAK

  tanggal:
    Jumlah sampel : 383
    Avg Sharpness : 20743.12
    Avg Confidence: 1.5%
    Status        : ✅ LAYAK


In [8]:
out_csv = os.path.join(BASE_DIR, "task3_evaluasi_awal.csv")
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n💾 Hasil disimpan: {out_csv}")
 
print(f"\n{'='*65}")
print("KESIMPULAN")
print(f"{'='*65}")
print("  Fokus utama MoneyLens:")
print("    → total_transaksi : nominal yang diinput user")
print("    → tanggal         : tanggal transaksi")
print()
print("  Jika sharpness kelas kritis < 50:")
print("    → Tingkatkan resolusi crop di Task 2")
print("    → Coba metode binarisasi berbeda")
print()
print("  Next step:")
print("    → Labeling teks ground truth per crop")
print("    → Training model dengan data berlabel")
print(f"{'='*65}")


💾 Hasil disimpan: D:\program vscode\MoneyLens\ai\Dataset_ocr\task3_evaluasi_awal.csv

KESIMPULAN
  Fokus utama MoneyLens:
    → total_transaksi : nominal yang diinput user
    → tanggal         : tanggal transaksi

  Jika sharpness kelas kritis < 50:
    → Tingkatkan resolusi crop di Task 2
    → Coba metode binarisasi berbeda

  Next step:
    → Labeling teks ground truth per crop
    → Training model dengan data berlabel
